## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Review scores to classes: equal width of 20 points

**Conversion to Categorical Variables** · a worked example from the StatNex teaching guide

**The story.** A company has 37 annual performance-review scores (0 to 100) and wants to turn them into letter grades for the bonus scheme.

**The question.** How many employees land in each grade, and how much do the cut points matter?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `12_utility/categorical_conversion.csv` (37 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Score
75.0
62.0
62.0
80.0
100.0
58.0
34.0
86.0
46.0
85.0
93.0
81.0
71.0
78.0
86.0
84.0
90.0
67.0
60.0
34.0
72.0
82.0
82.0
62.0
74.0
94.0
73.0
83.0
90.0
79.0
83.0
55.0
74.0
74.0
99.0
80.0
79.0
""")

## Libraries

In [ ]:
from decimal import Decimal

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
# Text and blank cells are left out of the statistics and get no category.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
column = "Score"
original = pd.to_numeric(data[column], errors="coerce")
values = original.dropna()

## Descriptive statistics

In [ ]:
summary = pd.Series({"n": len(values), "mean": values.mean(), "variance": values.var(ddof=1),
                     "SD": values.std(ddof=1), "minimum": values.min(), "maximum": values.max()})
print(summary)

## Boundaries and category names

In [ ]:
# Classes of equal width, starting at the minimum (the smallest value when left
# blank). Boundary i is minimum + i × width, rounded to the decimals the minimum and
# the width are written with: floating point makes 0.1 × 3 0.30000000000000004, which
# would put a value of 0.3 below its own boundary.
minimum = 20
if minimum is None:
    minimum = values.min()
width = float(20)


def decimals(x):
    """How many decimals x is written with, at 15 significant digits: 0.1 has 1, 20 has 0."""
    return max(0, -Decimal(f"{x:.15g}").as_tuple().exponent)


places = max(decimals(minimum), decimals(width))


def boundary(i):
    return round(float(minimum + i * width), places)


# As many classes as it takes to hold the largest value.
count = 1
# On a boundary it goes to the class above, so the top class starts at or below it.
while boundary(count) <= values.max():
    count += 1
boundaries = [boundary(i) for i in range(1, count)]
# Classes are numbered from the bottom: the lowest class is 1.
names = [str(i + 1) for i in range(count)]

# A value sitting exactly on a boundary goes to the class above it ("at least / less
# than"): every class is [lower, upper), closed on the left, which is right=False in
# pandas.cut below. On a grade boundary this decides somebody's grade.
right = False
legend = ([f"under {boundaries[0]:g}"] if boundaries else ["all cases"]) + [
    f"{low:g} to under {high:g}" for low, high in zip(boundaries, boundaries[1:])] + (
    [f"{boundaries[-1]:g} and over"] if boundaries else [])
print(pd.DataFrame({"range": legend, "category": names}).to_string(index=False))

## Converted data

In [ ]:
# pandas.cut sorts each value into its class. The outer edges are -inf and +inf, so a
# value below the first boundary or above the last still gets one; the boundaries
# must rise strictly. ordered=False lets two classes share a name.
classes = pd.cut(values, bins=[-np.inf, *boundaries, np.inf], right=right, labels=names,
                 ordered=False)
converted = classes.astype(object).reindex(data.index)
result = pd.DataFrame({column: original, "category": converted})
result.to_csv("categories.csv", index=False)
print()
print(result.to_string(index=False))

## Frequency distribution of the categories

In [ ]:
counts = converted.value_counts()
frequency = [int(counts.get(name, 0)) for name in names]
distribution = pd.DataFrame({"category": names, "frequency": frequency,
                             "relative frequency": np.array(frequency) / sum(frequency)})
print()
print(distribution.to_string(index=False))

## What the result says

This run cuts the scores into classes of equal width, 20 points each, and numbers them from 1 upward. The frequency distribution, the count in each class, gives 2 employees in class 1 (under 40), 3 in class 2, 15 in class 3 (60 to under 80), 16 in class 4 (80 to under 100) and 1 in class 5.

Class 5 holds only the one employee who scored exactly 100, because each class takes scores of at least its lower limit and less than its upper limit. Classes 3 and 4 hold 15 and 16 of the 37 employees, so the classes separate most of the staff very little.

Equal widths are simple to describe, but the cut points decide the grades here too.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.